# Semana 5 · ¿Por qué la Luna cambia de forma?

Este cuaderno acompaña a la sesión de proyecto de la semana 5: calcular la fase de la Luna de cualquier día y dibujarla.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar.

Tiempo estimado: 30-45 minutos.

## 1. El ciclo de fases

La Luna repite sus fases cada **mes sinódico**, de unos 29,53 días (de luna nueva a luna nueva). Si conocemos una luna nueva de referencia, la **edad** de la Luna (días desde la última luna nueva) en cualquier fecha es el resto de dividir el tiempo transcurrido entre ese periodo. Usamos la luna nueva media del 6 de enero de 2000 como referencia.

Es un modelo de **ciclo medio**: el movimiento real de la Luna no es uniforme y las fases reales se adelantan o retrasan hasta unas 14 horas respecto al modelo.

In [ ]:
from datetime import datetime, timezone
import numpy as np
import matplotlib.pyplot as plt

PERIODO = 29.530588861                                  # días: mes sinódico medio
REFERENCIA = datetime(2000, 1, 6, 18, 14, tzinfo=timezone.utc)   # luna nueva de referencia (aprox.)

def edad_luna(fecha):
    """Días desde la última luna nueva (modelo de ciclo medio). fecha: datetime con zona horaria UTC."""
    dias = (fecha - REFERENCIA).total_seconds() / 86400
    return dias % PERIODO

hoy = datetime.now(timezone.utc)
print(f"Edad de la Luna hoy (UTC): {edad_luna(hoy):.1f} días")

## 2. Elongación y fracción iluminada

La **elongación** $\varepsilon$ es el ángulo que separa la Luna del Sol visto desde la Tierra. Avanza 360° en un mes sinódico: $\varepsilon = 360^\circ \cdot \text{edad}/P$. La fracción del disco iluminada (de 0 a 1) es

$$k = \frac{1 - \cos\varepsilon}{2}$$

Todavía no has visto el coseno (semana 7): úsalo como caja negra. Comprueba que $\varepsilon = 0$ da $k = 0$ (luna nueva), $\varepsilon = 90^\circ$ da $k = 0{,}5$ y $\varepsilon = 180^\circ$ da $k = 1$ (luna llena).

In [ ]:
def elongacion(edad):
    return 360 * edad / PERIODO                # grados

def fraccion_iluminada(edad):
    return (1 - np.cos(np.radians(elongacion(edad)))) / 2

def nombre_fase(edad):
    f = edad / PERIODO
    nombres = [(0.03, "luna nueva"), (0.22, "creciente"), (0.28, "cuarto creciente"), (0.47, "gibosa creciente"),
               (0.53, "luna llena"), (0.72, "gibosa menguante"), (0.78, "cuarto menguante"), (0.97, "menguante"), (1.01, "luna nueva")]
    for limite, nombre in nombres:
        if f < limite:
            return nombre

for e in [0, 3.69, 7.38, 11.07, 14.77, 18.45, 22.15, 25.84]:
    print(f"edad {e:5.2f} d   elongación {elongacion(e):6.1f}°   iluminada {fraccion_iluminada(e)*100:5.1f} %   {nombre_fase(e)}")

Fíjate en que el **cuarto** creciente (a 7,4 días, una cuarta parte del ciclo) tiene **la mitad** del disco iluminada, no un cuarto. El nombre alude a la fracción del *ciclo*, no del disco.

## 3. Dibuja la fase

El borde iluminado es un semicírculo (el limbo) y la línea que separa luz y sombra, el **terminador**, es una semielipse. Los ejes son: el limbo de radio 1 y el terminador con ancho $\cos\varepsilon$ respecto al limbo. En el hemisferio norte la Luna creciente se ilumina por la derecha; en el sur, por la izquierda.

In [ ]:
def dibuja_luna(edad, ax, hemisferio_norte=True):
    eps = np.radians(elongacion(edad))
    t = np.linspace(0, np.pi, 200)
    creciente = eps <= np.pi
    e = eps if creciente else 2 * np.pi - eps
    limbo_x, y = np.sin(t), np.cos(t)
    term_x = np.cos(e) * np.sin(t)
    sentido = 1 if (creciente == hemisferio_norte) else -1       # a qué lado cae la parte iluminada
    ax.fill(np.r_[sentido * limbo_x, sentido * term_x[::-1]], np.r_[y, y[::-1]], color="#f1e9c8")
    ax.add_patch(plt.Circle((0, 0), 1, fill=False, color="gray"))
    ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.2, 1.2); ax.set_aspect("equal"); ax.axis("off")
    ax.set_facecolor("#1b2030")

fig, axs = plt.subplots(1, 8, figsize=(14, 2.2), facecolor="#1b2030")
for ax, e in zip(axs, np.arange(0, PERIODO, PERIODO / 8)):
    dibuja_luna(e, ax)
    ax.set_title(f"{e:.1f} d", color="white", fontsize=9)
plt.show()

## 4. La fase de cualquier día

Escribe la fecha que quieras y compara con lo que muestra Stellarium para esa misma fecha.

In [ ]:
fecha = datetime(2026, 12, 25, 12, 0, tzinfo=timezone.utc)    # cámbiala por cualquier otra
e = edad_luna(fecha)
print(f"{fecha:%d/%m/%Y}: edad {e:.1f} d, iluminada {fraccion_iluminada(e)*100:.0f} %, {nombre_fase(e)}")
fig, ax = plt.subplots(figsize=(2.5, 2.5), facecolor="#1b2030")
dibuja_luna(e, ax)
plt.show()

## 5. Cuándo es la próxima luna llena

La luna llena ocurre a media edad del ciclo (la mitad del periodo). Si hoy la edad es $e$ días, faltan $P/2 - e$ días para la próxima llena (si $e < P/2$) o $3P/2 - e$ si ya pasó.

In [ ]:
from datetime import timedelta

def proxima_llena(fecha):
    e = edad_luna(fecha)
    faltan = PERIODO / 2 - e if e < PERIODO / 2 else 1.5 * PERIODO - e
    return fecha + timedelta(days=faltan)

print("Próxima luna llena (modelo medio, ±1 día):", proxima_llena(hoy).strftime("%d/%m/%Y"))

## 6. Para pensar

1. Compara el resultado del modelo con Stellarium para cinco fechas distintas. ¿Cuál es el desfase máximo? ¿A qué lo atribuyes?
2. ¿Cómo cambia el dibujo para un observador del hemisferio sur? Cambia `hemisferio_norte` a `False` y compruébalo.
3. ¿Qué elongación tiene la Luna cuando sale al anochecer?